# Cluster Setup Guide

## Prerequisites
- Databricks workspace (Free Edition or higher)
- Cloudflare R2 account with bucket `insee-sirene-monitor-data-landing-zone`
- GitHub account with access to this repo

## Step 1 — Link GitHub repo
1. Databricks → User Settings → Linked accounts → GitHub
2. Generate a GitHub Classic token with `repo` scope
3. Connect and clone this repo into your Workspace

## Step 2a — Configure R2 secrets
Run the following in a temporary notebook (delete after use — never commit credentials):


In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import requests

DATABRICKS_HOST = "https://YOUR-WORKSPACE.cloud.databricks.com"
DATABRICKS_TOKEN = "YOUR-DATABRICKS-TOKEN"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

# Create scope
requests.post(f"{DATABRICKS_HOST}/api/2.0/secrets/scopes/create", headers=headers,
    json={"scope": "r2-insee-sirene-monitor-dlz-credentials"})

# Add secrets
for key, value in [
    ("access_key", "YOUR_R2_ACCESS_KEY"),
    ("secret_key", "YOUR_R2_SECRET_KEY")
]:
    requests.post(f"{DATABRICKS_HOST}/api/2.0/secrets/put", headers=headers,
        json={"scope": "r2-insee-sirene-monitor-dlz-credentials", "key": key, "string_value": value})

print("Secrets configured")

## Step 2b — Configure INSEE API credentials

Get your INSEE API key at: https://portail-api.insee.fr
- Create a new "simple" application (not "backend to backend")
- Go to "Souscriptions" tab
- Subscribe to "Sirene V3" API
- Got to your app space, "Souscriptions" tab, select the API, and copy the API Key displayed on the right part of the screen

Run in the same temporary notebook as Step 2a:

In [0]:
# Create scope
requests.post(f"{DATABRICKS_HOST}/api/2.0/secrets/scopes/create", headers=headers,
    json={"scope": "insee-sirene-monitor-api-credentials"})

# Add API key
requests.post(f"{DATABRICKS_HOST}/api/2.0/secrets/put", headers=headers,
    json={
        "scope": "insee-sirene-monitor-api-credentials",
        "key": "api_key",
        "string_value": "YOUR_INSEE_API_KEY"
    })

print("INSEE credentials configured")

## Step 3 — Install package on cluster
Run in a notebook attached to your cluster:

In [0]:
%sh
pip install -e /Workspace/Users/YOUR_USER/insee-sirene-monitor

## Step 4a — Verify R2 connection
Run in a notebook attached to your cluster:

In [0]:
from utils.storage import get_r2_client, is_first_run

client = get_r2_client(dbutils)

print("First run:", is_first_run(client))

Expected output: `First run: True`

## Step 4b — Verify INSEE API connection
Run in a notebook attached to your cluster:

In [0]:
from utils.storage import get_insee_api_key
import requests

api_key = get_insee_api_key(dbutils)

response = requests.get(
    "https://api.insee.fr/api-sirene/3.11/siren/309634954",
    headers={"X-INSEE-Api-Key-Integration": api_key}
)
print("INSEE API OK:", response.status_code == 200)